In [ ]:
import re, warnings, math, time
import numpy as np
warnings.filterwarnings('ignore')

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches

from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel

try:
    from qiskit_ibm_runtime.fake_provider import FakeSherbrooke
except ImportError:
    from qiskit.providers.fake_provider import FakeSherbrooke

# ── Constants ─────────────────────────────────────────────────────────────────
with open("sequence.txt", "r") as f:
    DNA = f.read().strip()

WINDOW_SIZE  = 3
N_QUBITS     = WINDOW_SIZE * 2
WAVE_PHI     = np.pi / 4
SHOTS        = 2048

BASE_MAP     = {'A': 0, 'T': 1, 'G': 2, 'C': 3}
IDX_TO_BASE  = {v: k for k, v in BASE_MAP.items()}
NATIVE_GATES = ['x', 'rz', 'sx', 'ry', 'cx', 'reset', 'measure', 'id']
BASE_RY      = {'A': 0, 'T': 1, 'G': 1, 'C': 2}


# ═════════════════════════════════════════════════════════════════════════════
# SECTION 1 – Sequence utilities
# ═════════════════════════════════════════════════════════════════════════════
def clean_dna(s):
    return re.sub(r'[^ATGC]', '', s.upper())

def make_windows(seq, w):
    chunks    = [seq[i:i+w] for i in range(0, len(seq), w)]
    real_lens = [len(c) for c in chunks]
    padded    = [c + 'A' * (w - len(c)) for c in chunks]
    return padded, real_lens


# ═════════════════════════════════════════════════════════════════════════════
# SECTION 2 – Ry encoding
# ═════════════════════════════════════════════════════════════════════════════
def ry_encode_base(qc, base, msb_qubit, lsb_qubit):
    val = BASE_MAP[base]
    if val & 2:
        qc.ry(np.pi, msb_qubit)
    if val & 1:
        qc.ry(np.pi, lsb_qubit)

def ry_encode_window(qc, bases):
    for i, base in enumerate(bases):
        ry_encode_base(qc, base, msb_qubit=i*2, lsb_qubit=i*2+1)


# ═════════════════════════════════════════════════════════════════════════════
# SECTION 3 – Wave filter
# ═════════════════════════════════════════════════════════════════════════════
def apply_wave_filter(qc, phi, n_bases):
    for i in range(n_bases - 1):
        qc.cry(phi, i * 2 + 1, i * 2 + 2)

def apply_inverse_wave_filter(qc, phi, n_bases):
    for i in reversed(range(n_bases - 1)):
        qc.cry(-phi, i * 2 + 1, i * 2 + 2)


# ═════════════════════════════════════════════════════════════════════════════
# SECTION 4 – Circuit builders
# ═════════════════════════════════════════════════════════════════════════════
def build_wave_window_circuit(bases, phi=WAVE_PHI, measure=True):
    rlen = len(bases)
    n_q  = rlen * 2
    qc   = QuantumCircuit(n_q, n_q if measure else 0,
                          name=f"RyWave({''.join(bases)})")
    ry_encode_window(qc, bases)
    if rlen > 1:
        apply_wave_filter(qc, phi, rlen)
        apply_inverse_wave_filter(qc, phi, rlen)
    if measure:
        qc.measure(range(n_q), range(n_q))
    return qc

def build_fidelity_circuit(bases, phi=WAVE_PHI):
    return build_wave_window_circuit(bases, phi=phi, measure=False)


# ═════════════════════════════════════════════════════════════════════════════
# SECTION 5 – Decoder
# ═════════════════════════════════════════════════════════════════════════════
def decode_bitstring(bitstring, n_bases):
    bits  = bitstring[::-1]
    bases = []
    for i in range(n_bases):
        msb = int(bits[i * 2])
        lsb = int(bits[i * 2 + 1])
        bases.append(IDX_TO_BASE[msb * 2 + lsb])
    return ''.join(bases)

def recover_sequence(all_counts, real_lens):
    recovered  = ''
    per_window = []
    for counts, rlen in zip(all_counts, real_lens):
        best    = max(counts, key=counts.get)
        decoded = decode_bitstring(best, rlen)
        recovered += decoded
        per_window.append(decoded)
    return recovered, per_window


# ═════════════════════════════════════════════════════════════════════════════
# SECTION 6 – Simulation
# ═════════════════════════════════════════════════════════════════════════════
def simulate_windows(windows, real_lens, noisy=False, shots=SHOTS,
                     noise_model=None):
    """Run all window circuits on either the ideal or noisy simulator."""
    sim = (AerSimulator(method='density_matrix', noise_model=noise_model)
           if noisy else AerSimulator(method='statevector'))

    all_counts = []
    for window, rlen in zip(windows, real_lens):
        qc   = build_wave_window_circuit(window[:rlen], phi=WAVE_PHI, measure=True)
        t_qc = transpile(qc, basis_gates=NATIVE_GATES,
                         coupling_map=None, optimization_level=3,
                         seed_transpiler=42)
        all_counts.append(sim.run(t_qc, shots=shots).result().get_counts())
    return all_counts


def _target_index(bases):
    """
    Compute the expected computational basis state index after
    encode + wave + inverse (= identity), using Qiskit's qubit-0-as-LSB convention.
    """
    idx = 0
    for i, base in enumerate(bases):
        val = BASE_MAP[base]
        idx |= (((val >> 1) & 1) << (i * 2))
        idx |= ((val & 1)        << (i * 2 + 1))
    return idx


def compute_fidelity_fast(window, sim_noisy, noise_model):
    """
    State fidelity via the diagonal trick.

    The ideal output is a pure computational basis state |ψ⟩, so:
        F = ⟨ψ|ρ_noisy|ψ⟩ = ρ_noisy[idx, idx]
    Only one density-matrix simulation is needed.
    """
    qc   = build_fidelity_circuit(window, phi=WAVE_PHI)
    t_qc = transpile(qc,
                     basis_gates=list(FakeSherbrooke().operation_names),
                     coupling_map=None, optimization_level=3,
                     seed_transpiler=42)

    qc_dm = t_qc.copy()
    qc_dm.save_density_matrix()
    dm    = np.array(sim_noisy.run(qc_dm, shots=1).result()
                               .data()['density_matrix'])

    idx      = _target_index(window)
    fidelity = float(np.real(dm[idx, idx]))
    depth    = t_qc.depth()
    gates    = t_qc.size()
    n_2q     = sum(1 for inst in t_qc.data if len(inst.qubits) == 2)
    n_swap   = sum(1 for inst in t_qc.data if inst.operation.name == 'swap')
    return fidelity, depth, gates, n_2q, n_swap


def compute_all_fidelities(windows, real_lens, backend):
    """
    Compute fidelity for all unique windows.
    Noise model and simulator are built once and shared.
    """
    noise_model = NoiseModel.from_backend(backend)
    sim_noisy   = AerSimulator(method='density_matrix', noise_model=noise_model)

    seen     = {}
    fids     = []
    n_unique = len({window[:rlen] for window, rlen in zip(windows, real_lens)})
    done     = 0

    print(f"\n  Computing fidelity for {n_unique} unique windows…")
    for window, rlen in zip(windows, real_lens):
        key = window[:rlen]
        if key not in seen:
            seen[key] = compute_fidelity_fast(key, sim_noisy, noise_model)
            done += 1
            pct = done / n_unique * 100
            print(f'\r  [{done:>3}/{n_unique}  {pct:5.1f}%]  '
                  f'{"█" * (done * 20 // n_unique):<20}',
                  end='', flush=True)
        fids.append(seen[key][0])
    print()

    worst                        = min(seen, key=lambda k: seen[k][0])
    _, depth, gates, n_2q, n_swap = seen[worst]
    return fids, float(np.mean(fids)), float(np.min(fids)), depth, gates, n_2q, n_swap


# ═════════════════════════════════════════════════════════════════════════════
# SECTION 7 – Metrics
# ═════════════════════════════════════════════════════════════════════════════
def count_gates(qc):
    counts = {}
    for inst in qc.data:
        counts[inst.operation.name] = counts.get(inst.operation.name, 0) + 1
    return counts

def sequence_accuracy(orig, rec):
    assert len(orig) == len(rec)
    return sum(a == b for a, b in zip(orig, rec)) / len(orig)


# ═════════════════════════════════════════════════════════════════════════════
# SECTION 8 – Diagram generation
# ═════════════════════════════════════════════════════════════════════════════
def generate_results_dashboard(seq, windows, real_lens,
                                all_fids, mean_f, min_f,
                                rec_ideal, rec_noisy,
                                acc_ideal, acc_noisy,
                                depth, n_2q, n_swap,
                                ry_used, ry_max):

    BG     = '#0d1117'; CARD = '#161b22'; BRD = '#30363d'
    TXT    = '#e6edf3'; DIM  = '#8b949e'
    BLU    = '#58a6ff'; GRN  = '#3fb950'
    ORG    = '#d29922'; RED  = '#f85149'
    BASE_C = {'A': GRN, 'T': ORG, 'G': BLU, 'C': RED}

    fig = plt.figure(figsize=(18, 11), facecolor=BG)
    gs  = gridspec.GridSpec(2, 3, figure=fig,
                            hspace=0.48, wspace=0.36,
                            left=0.05, right=0.97, top=0.91, bottom=0.06)

    fig.text(0.5, 0.96,
             'Ry-Wave Filter — Quantum DNA Encoding Results',
             color=TXT, fontsize=13, fontweight='bold', ha='center')
    fig.text(0.5, 0.93,
             f"Sequence: {seq[:40]}{'…' if len(seq)>40 else ''}  "
             f"({len(seq)} bp)  |  {N_QUBITS} qubits  |  {len(windows)} windows",
             color=DIM, fontsize=9, ha='center')

    def mkax(loc, title):
        ax = fig.add_subplot(loc)
        ax.set_facecolor(CARD)
        for s in ax.spines.values():
            s.set_color(BRD); s.set_linewidth(1.1)
        ax.tick_params(colors=DIM, labelsize=8)
        ax.xaxis.label.set_color(DIM)
        ax.yaxis.label.set_color(DIM)
        ax.set_title(title, color=TXT, fontsize=9,
                     fontweight='bold', pad=7, loc='left')
        return ax

    # Panel 1 – per-window fidelity bar chart
    ax_f = mkax(gs[0, 0], 'Per-window state fidelity  (FakeSherbrooke)')
    bar_colors = [GRN if f >= 0.90 else (ORG if f >= 0.80 else RED)
                  for f in all_fids]
    ax_f.bar(range(len(all_fids)), all_fids,
             color=bar_colors, width=0.8, linewidth=0, zorder=3)
    ax_f.axhline(mean_f, color=BLU, lw=1.5, ls='--',
                 label=f'Mean  {mean_f:.4f}', zorder=4)
    ax_f.axhline(min_f,  color=ORG, lw=1.0, ls=':',
                 label=f'Min   {min_f:.4f}', zorder=4)
    ax_f.set_ylim(max(0, min_f - 0.05), 1.02)
    ax_f.set_xlabel('Window index', fontsize=8)
    ax_f.set_ylabel('Fidelity', fontsize=8)
    ax_f.legend(fontsize=7.5, facecolor=CARD, edgecolor=BRD, labelcolor=DIM)
    ax_f.yaxis.grid(True, color=BRD, lw=0.5)
    ax_f.set_axisbelow(True)

    # Panel 2 – fidelity gauge
    ax_g = mkax(gs[0, 1], 'Mean fidelity gauge')
    ax_g.axis('off')
    for t0, t1, c in [(0, np.pi*0.33, RED),
                      (np.pi*0.33, np.pi*0.66, ORG),
                      (np.pi*0.66, np.pi, GRN)]:
        t = np.linspace(t0, t1, 100)
        ax_g.plot(np.cos(t), np.sin(t), lw=18, color=c, alpha=0.25,
                  solid_capstyle='round')
        ax_g.plot(np.cos(t), np.sin(t), lw=2,  color=c, alpha=0.7)
    needle = mean_f * np.pi
    ax_g.annotate('', xy=(0.78*np.cos(needle), 0.78*np.sin(needle)),
                  xytext=(0, 0),
                  arrowprops=dict(arrowstyle='->', color=TXT, lw=2.8))
    ax_g.plot(0, 0, 'o', color=TXT, ms=7, zorder=10)
    ax_g.text(0,  0.22, f'{mean_f:.4f}', color=GRN,
              fontsize=22, fontweight='bold', ha='center')
    ax_g.text(0, -0.06, f'{mean_f*100:.2f}%  mean fidelity',
              color=DIM, fontsize=8.5, ha='center')
    ax_g.text(-0.90, 0.04, '0', color=DIM, fontsize=8)
    ax_g.text( 0.80, 0.04, '1', color=DIM, fontsize=8)
    ax_g.set_xlim(-1.15, 1.15)
    ax_g.set_ylim(-0.38, 1.18)

    # Panel 3 – judging-criteria scorecard
    ax_s = mkax(gs[0, 2], 'Judging criteria — circuit metrics')
    ax_s.axis('off')
    rows = [
        ('Qubits used',     f'{N_QUBITS}',          f'vs {len(seq)*2} naive  ({len(seq)*2//N_QUBITS}× reduction)'),
        ('State fidelity',  f'{mean_f:.4f}',         f'mean  |  min {min_f:.4f}'),
        ('Circuit depth',   f'{depth}',              'post-transpile (worst window)'),
        ('2Q gate count',   f'{n_2q}',               'CX equivalents / window'),
        ('SWAP gate count', f'{n_swap}',             'post-transpile / window'),
        ('Ideal recovery',  f'{acc_ideal*100:.1f}%', 'AerSimulator statevector'),
        ('Noisy recovery',  f'{acc_noisy*100:.1f}%', 'FakeSherbrooke density matrix'),
        ('Ry gates saved',  f'{ry_max - ry_used}',   'A-bases cost 0 gates each'),
        ('Encoding method', 'Ry-Wave  φ=π/4',        'window size 3 bp'),
    ]
    y0 = 0.95; rh = 0.098
    for ri, (label, val, note) in enumerate(rows):
        y_ = y0 - ri * rh
        ax_s.text(0.02, y_, label, color=DIM,  fontsize=8,   va='center')
        ax_s.text(0.45, y_, val,   color=BLU,  fontsize=8.5, va='center',
                  fontweight='bold')
        ax_s.text(0.68, y_, note,  color=DIM,  fontsize=7,   va='center')
    ax_s.set_xlim(0, 1); ax_s.set_ylim(0, 1)

    # Panel 4 – fidelity distribution histogram
    ax_h = mkax(gs[1, 0], 'Fidelity distribution')
    bins = min(15, max(5, len(set(round(f, 3) for f in all_fids))))
    ax_h.hist(all_fids, bins=bins, color=BLU, alpha=0.75,
              edgecolor=BRD, linewidth=0.8, zorder=3)
    ax_h.axvline(mean_f, color=GRN, lw=1.8, ls='--', label=f'Mean {mean_f:.3f}')
    ax_h.axvline(min_f,  color=ORG, lw=1.2, ls=':',  label=f'Min  {min_f:.3f}')
    ax_h.set_xlabel('Fidelity', fontsize=8)
    ax_h.set_ylabel('Count', fontsize=8)
    ax_h.legend(fontsize=7.5, facecolor=CARD, edgecolor=BRD, labelcolor=DIM)
    ax_h.yaxis.grid(True, color=BRD, lw=0.5)
    ax_h.set_axisbelow(True)

    # Panel 5 – scalability projection
    ax_sc = mkax(gs[1, 1], 'Scalability projection')
    ax_sc.axis('off')
    for label, x in [('Length (bp)', 0.02), ('Windows', 0.42),
                      ('Qubits', 0.62), ('Est. F', 0.80)]:
        ax_sc.text(x, 0.95, label, color=TXT, fontsize=8,
                   fontweight='bold', va='center')
    eps_ry = 0.001; eps_cx = 0.0091
    avg_ry = ry_used / len(windows)
    for ri, length in enumerate([50, 1_000, 12_000, 3_000_000_000]):
        y_ = 0.82 - ri * 0.15
        w  = math.ceil(length / WINDOW_SIZE)
        F  = np.exp(-(n_2q * eps_cx) - (avg_ry * eps_ry))
        ax_sc.text(0.02, y_, f'{length:,}',  color=BLU, fontsize=8, va='center')
        ax_sc.text(0.42, y_, f'{w:,}',       color=DIM, fontsize=8, va='center')
        ax_sc.text(0.62, y_, f'{N_QUBITS}',  color=GRN, fontsize=8, va='center')
        ax_sc.text(0.80, y_, f'{F:.3f}',     color=ORG, fontsize=8, va='center')
    ax_sc.set_xlim(0, 1); ax_sc.set_ylim(0, 1)

    # Panel 6 – decoded strand comparison (first 50 bases)
    SHOW  = 50
    seq_d = seq[:SHOW]; rid_d = rec_ideal[:SHOW]; rno_d = rec_noisy[:SHOW]
    ax_str = mkax(gs[1, 2], f'Decoded strand — first {len(seq_d)} bases')
    ax_str.set_xlim(-1.0, len(seq_d) - 0.5)
    ax_str.set_ylim(-0.6, 2.6)
    ax_str.axis('off')
    for row_y, label, strand in zip(
            [2.0, 1.1, 0.2],
            ['Original', 'Ideal', 'Noisy'],
            [seq_d, rid_d, rno_d]):
        ax_str.text(-0.8, row_y, label, color=TXT, fontsize=7,
                    va='center', ha='right', fontweight='bold')
        for i, base in enumerate(strand):
            c      = BASE_C.get(base, DIM)
            is_err = (label == 'Noisy' and base != seq_d[i])
            ax_str.add_patch(mpatches.FancyBboxPatch(
                (i - 0.46, row_y - 0.40), 0.92, 0.80,
                boxstyle='round,pad=0.04',
                facecolor=(RED if is_err else CARD),
                edgecolor=(RED if is_err else c),
                linewidth=(2.0 if is_err else 0.7), zorder=3))
            ax_str.text(i, row_y, base,
                        color=(TXT if is_err else c),
                        fontsize=5.5, ha='center', va='center',
                        fontweight='bold', zorder=4)
    for row_y, acc in zip([1.1, 0.2], [acc_ideal, acc_noisy]):
        c_ = GRN if acc == 1.0 else (ORG if acc >= 0.95 else RED)
        ax_str.text(len(seq_d) + 0.5, row_y, f'{acc*100:.1f}%',
                    color=c_, fontsize=9, va='center', fontweight='bold')
    for base, c in BASE_C.items():
        ax_str.scatter([], [], color=c, s=50, label=base, marker='s')
    ax_str.legend(loc='upper right', fontsize=7, facecolor=CARD,
                  edgecolor=BRD, labelcolor=DIM, ncol=4,
                  markerscale=1.1, framealpha=0.9)

    fig.savefig('results_dashboard.png', dpi=160,
                bbox_inches='tight', facecolor=BG)
    plt.close(fig)
    print("  Saved: results_dashboard.png")


# ═════════════════════════════════════════════════════════════════════════════
# MAIN
# ═════════════════════════════════════════════════════════════════════════════
def main():
    seq                = clean_dna(DNA)
    windows, real_lens = make_windows(seq, WINDOW_SIZE)
    backend            = FakeSherbrooke()
    noise_model        = NoiseModel.from_backend(backend)

    print(f"Sequence : {seq[:60]}{'…' if len(seq)>60 else ''}")
    print(f"Length   : {len(seq)} bp  |  Windows: {len(windows)}  |  Qubits: {N_QUBITS}")

    # ── Circuit anatomy (representative window) ────────────────────────
    rep_win = windows[1][:real_lens[1]]
    qc_rep  = build_wave_window_circuit(rep_win, phi=WAVE_PHI, measure=True)
    t_rep   = transpile(qc_rep, basis_gates=NATIVE_GATES,
                        coupling_map=None, optimization_level=3,
                        seed_transpiler=42)
    tc = count_gates(t_rep)
    print(f"\nCircuit (window '{rep_win}', post-transpile):")
    print(f"  Depth : {t_rep.depth()}")
    print(f"  CX    : {tc.get('cx', 0)}")
    print(f"  SWAP  : {tc.get('swap', 0)}")
    print(f"  Rz    : {tc.get('rz', 0)}  SX: {tc.get('sx', 0)}  Ry: {tc.get('ry', 0)}")

    ry_used = sum(BASE_RY.get(b, 0) for b in seq)
    ry_max  = len(seq) * 2
    print(f"\nRy pruning: {ry_used}/{ry_max} used  ({ry_max - ry_used} saved via A=|00⟩)")

    # ── State fidelity (FakeSherbrooke noisy vs ideal basis state) ─────
    t0 = time.time()
    all_fids, mean_f, min_f, depth, gates, n_2q, n_swap = \
        compute_all_fidelities(windows, real_lens, backend)
    print(f"  done in {time.time()-t0:.1f}s")
    print(f"  Mean fidelity  : {mean_f:.6f}  ({mean_f*100:.2f}%)")
    print(f"  Min  fidelity  : {min_f:.6f}  ({min_f*100:.2f}%)")
    print(f"  Depth / win    : {depth}")
    print(f"  2Q gates / win : {n_2q}")
    print(f"  SWAP / win     : {n_swap}")

    # ── Ideal recovery (AerSimulator statevector) ──────────────────────
    t0           = time.time()
    ideal_counts = simulate_windows(windows, real_lens, noisy=False,
                                    noise_model=noise_model)
    rec_ideal, _ = recover_sequence(ideal_counts, real_lens)
    acc_ideal    = sequence_accuracy(seq, rec_ideal)
    print(f"\nIdeal recovery  ({time.time()-t0:.1f}s): {acc_ideal*100:.2f}%"
          f"  ({'PERFECT' if acc_ideal == 1.0 else 'PARTIAL'})")

    # ── Noisy recovery (FakeSherbrooke) ───────────────────────────────
    t0            = time.time()
    noisy_counts  = simulate_windows(windows, real_lens, noisy=True,
                                     noise_model=noise_model)
    rec_noisy, _  = recover_sequence(noisy_counts, real_lens)
    acc_noisy     = sequence_accuracy(seq, rec_noisy)
    print(f"Noisy recovery  ({time.time()-t0:.1f}s): {acc_noisy*100:.2f}%")

    # ── Scalability table ─────────────────────────────────────────────
    eps_ry = 0.001; eps_cx = 0.0091
    avg_ry = ry_used / len(windows)
    print(f"\n{'─'*56}")
    print(f"  {'Length (bp)':>18}  {'Qubits':>6}  {'Windows':>9}  {'2Q/win':>6}  {'Est.F':>6}")
    print(f"{'─'*56}")
    for length in [50, 1_000, 12_000, 3_000_000_000]:
        w = math.ceil(length / WINDOW_SIZE)
        F = np.exp(-(n_2q * eps_cx) - (avg_ry * eps_ry))
        print(f"  {length:>18,}  {N_QUBITS:>6}  {w:>9,}  {n_2q:>6}  {F:>6.3f}")
    print(f"{'─'*56}")

    # ── Final scorecard ───────────────────────────────────────────────
    print(f"\n{'═'*56}")
    print(f"  Qubits used    : {N_QUBITS}  (naive = {len(seq)*2},  {len(seq)*2//N_QUBITS}× reduction)")
    print(f"  State fidelity : {mean_f:.6f}  mean  |  {min_f:.6f}  min")
    print(f"  Circuit depth  : {depth}  (post-transpile, worst window)")
    print(f"  2Q gates / win : {n_2q}")
    print(f"  SWAP / win     : {n_swap}")
    print(f"  Ideal accuracy : {acc_ideal*100:.2f}%")
    print(f"  Noisy accuracy : {acc_noisy*100:.2f}%")
    print(f"{'═'*56}")

    # ── Dashboard ─────────────────────────────────────────────────────
    generate_results_dashboard(
        seq=seq, windows=windows, real_lens=real_lens,
        all_fids=all_fids, mean_f=mean_f, min_f=min_f,
        rec_ideal=rec_ideal, rec_noisy=rec_noisy,
        acc_ideal=acc_ideal, acc_noisy=acc_noisy,
        depth=depth, n_2q=n_2q, n_swap=n_swap,
        ry_used=ry_used, ry_max=ry_max,
    )

    return dict(
        n_qubits=N_QUBITS,
        fidelity_mean=mean_f,     fidelity_min=min_f,
        circuit_depth=depth,      cx_per_window=n_2q,
        swap_per_window=n_swap,
        ideal_accuracy=acc_ideal, noisy_accuracy=acc_noisy,
        original=seq, recovered_ideal=rec_ideal, recovered_noisy=rec_noisy,
    )


if __name__ == "__main__":
    main()
